# Post-Engineering Exploratory Data Analysis (EDA)
---
## Tasks done in this notebook:
- Load engineered features and target labels
- Bivariate analysis: Engineered numerical features vs Target
- Bivariate analysis: Engineered categorical/flag features vs Target
- Multivariate analysis involving the Target (is_fraud)
- Correlation analysis of full engineered feature space
- EDA Summary & Modeling Hypotheses

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from itertools import combinations
import warnings as warn
warn.filterwarnings('ignore')

sns.set_style('whitegrid')
sns.set_palette('pastel')
fraud_colors = ['lightblue', 'salmon']

### Data Loading and First Look

In [ ]:
X_train = pd.read_csv('../data/interim/unscaled/X_train.csv')
Y_train = pd.read_csv('../data/interim/label/Y_train.csv')

X_train.columns = X_train.columns.str.strip()
Y_train.columns = Y_train.columns.str.strip()

train_df = pd.concat([X_train, Y_train], axis=1)

print("ENGINEERED TRAIN DATASET SNAPSHOT")
print(f"Shape: {train_df.shape}")
display(train_df.head(2))
print("\nDataset Info:")
train_df.info()

fraud_rate = train_df['is_fraud'].mean()
print(f"\nTrain Fraud Rate: {fraud_rate:.4f}")
print(f"Train Class Distribution:\n{train_df['is_fraud'].value_counts(normalize=True)}")

### Bivariate Analysis: Engineered Numerical Features vs Target
---
We examine how transformed numerical features separate fraud from legitimate transactions using box plots, violin plots, and KDE density comparisons.

In [ ]:
engineered_num_cols = ['log_amount', 'velocity_per_hour', 'relevant_amount']

for col in engineered_num_cols:
    print(f"\n{col} vs is_fraud")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    sns.boxplot(data=train_df, x='is_fraud', y=col, ax=axes[0], palette=fraud_colors, hue='is_fraud', showfliers=False)
    axes[0].set_title(f'Box Plot: {col} by Fraud Status', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Is Fraud (0=No, 1=Yes)')
    axes[0].set_ylabel(col)
    axes[0].legend().remove()
    
    sns.violinplot(data=train_df, x='is_fraud', y=col, ax=axes[1], palette=fraud_colors, hue='is_fraud', inner='quartile')
    axes[1].set_title(f'Violin Plot: {col} by Fraud Status', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Is Fraud (0=No, 1=Yes)')
    axes[1].set_ylabel(col)
    axes[1].legend().remove()
    
    sns.kdeplot(data=train_df, x=col, hue='is_fraud', fill=True, common_norm=False, palette=fraud_colors, alpha=0.4, ax=axes[2])
    axes[2].set_title(f'KDE Density: {col} by Fraud Status', fontsize=12, fontweight='bold')
    axes[2].set_xlabel(col)
    axes[2].set_ylabel('Density')
    
    plt.tight_layout()
    plt.show()
    
    summary = train_df.groupby('is_fraud')[col].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    print(f"Statistical Summary for {col}:\n{summary}\n")

### Bivariate Analysis: Engineered Binary Flags (Categorical) vs Target
---
We analyze engineered boolean flags and original binary signals to quantify their lift in fraud probability.

In [ ]:
flag_cols = ['is_night_transaction', 'high_risk_abroad', 'is_high_velocity_low_trust']

for col in flag_cols:
    plt.figure(figsize=(8, 5))
    fraud_by_flag = train_df.groupby(col)['is_fraud'].mean().reset_index()
    fraud_by_flag.columns = [col, 'fraud_rate']
    
    sns.barplot(x=col, y='fraud_rate', data=fraud_by_flag, palette=fraud_colors, hue=col, legend=False)
    plt.title(f'Fraud Rate by {col}')
    plt.xlabel(col)
    plt.ylabel('Fraud Rate')
    plt.xticks([0, 1], ['No', 'Yes'])
    
    max_rate = fraud_by_flag['fraud_rate'].max() if len(fraud_by_flag) > 0 else 0.05
    plt.ylim(0, max_rate * 1.3)
    plt.show()

### Bivariate Analysis: One-Hot Encoded Merchant Categories vs Target
---
We examine fraud rates across different merchant categories after one-hot encoding. Clothing serves as the reference category.

In [ ]:
ohe_cols = ['merchant_category_Electronics', 'merchant_category_Food', 'merchant_category_Grocery', 'merchant_category_Travel']

plt.figure(figsize=(12, 6))

category_fraud_rates = []
category_names = []

for col in ohe_cols:
    cat_name = col.replace('merchant_category_', '')
    fraud_rate = train_df[train_df[col] == 1]['is_fraud'].mean()
    category_fraud_rates.append(fraud_rate)
    category_names.append(cat_name)

clothing_mask = (train_df[ohe_cols] == 0).all(axis=1)
clothing_fraud_rate = train_df[clothing_mask]['is_fraud'].mean()
category_fraud_rates.append(clothing_fraud_rate)
category_names.append('Clothing (Reference)')

cat_df = pd.DataFrame({'Category': category_names, 'Fraud_Rate': category_fraud_rates}).sort_values('Fraud_Rate', ascending=True)

ax = sns.barplot(data=cat_df, y='Category', x='Fraud_Rate', palette='viridis')

for i, row in cat_df.iterrows():
    ax.text(row['Fraud_Rate'] + 0.001, i, f"{row['Fraud_Rate']*100:.2f}%", va='center', fontsize=9)

plt.title('Fraud Rate by Merchant Category', fontsize=14, fontweight='bold')
plt.xlabel('Fraud Rate')
plt.ylabel('Merchant Category')
plt.xlim(0, cat_df['Fraud_Rate'].max() * 1.2)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("Merchant Category Fraud Rates (sorted):")
for _, row in cat_df.iterrows():
    print(f"  {row['Category']}: {row['Fraud_Rate']*100:.2f}%")

### Multivariate Analysis: Feature Interactions with Target
---
We test whether engineered features interact synergistically. High fraud often emerges from combinations (e.g., high velocity + low trust, or large amounts at night).

In [ ]:
sample_df = train_df.sample(min(3000, len(train_df)), random_state=42)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=sample_df, x='log_amount', y='velocity_per_hour', hue='is_fraud', palette=fraud_colors, alpha=0.5, s=30)
plt.title('Log Amount vs Velocity per Hour (Colored by Fraud)', fontsize=13, fontweight='bold')
plt.xlabel('Log Amount')
plt.ylabel('Velocity per Hour')
plt.legend(title='Is Fraud', labels=['Legit', 'Fraud'])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
sns.scatterplot(data=sample_df, x='device_trust_score', y='relevant_amount', hue='is_fraud', palette=fraud_colors, alpha=0.5, s=30)
plt.title('Device Trust Score vs Relevant Amount (Colored by Fraud)', fontsize=13, fontweight='bold')
plt.xlabel('Device Trust Score')
plt.ylabel('Relevant Amount (Normalized)')
plt.legend(title='Is Fraud', labels=['Legit', 'Fraud'])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("\nConditional Fraud Rate Analysis")

train_df['night_high_vel'] = ((train_df['is_night_transaction'] == 1) & (train_df['velocity_last_24h'] >= train_df['velocity_last_24h'].quantile(0.75))).astype(int)

conditional_fraud = train_df.groupby('night_high_vel')['is_fraud'].agg(['count', 'mean', 'sum'])
conditional_fraud.columns = ['Total', 'Fraud_Rate', 'Fraud_Count']
conditional_fraud['Fraud_Rate_Pct'] = conditional_fraud['Fraud_Rate'] * 100

print("\nFraud Rate: Night Transaction + High Velocity (75th percentile)")
print(conditional_fraud[['Total', 'Fraud_Count', 'Fraud_Rate_Pct']])

plt.figure(figsize=(8, 5))
sns.barplot(x=conditional_fraud.index.astype(str), y=conditional_fraud['Fraud_Rate_Pct'], palette=['lightblue', 'salmon'], hue=conditional_fraud.index.astype(str), legend=False)
plt.title('Fraud Rate: Night + High Velocity Interaction', fontsize=13, fontweight='bold')
plt.xlabel('Night & High Velocity Flag (0=No, 1=Yes)')
plt.ylabel('Fraud Rate (%)')
plt.xticks([0, 1], ['Normal Conditions', 'Night + High Velocity'])
plt.grid(axis='y', alpha=0.3)

for i, row in conditional_fraud.iterrows():
    plt.text(i, row['Fraud_Rate_Pct'] + 0.3, f"{row['Fraud_Rate_Pct']:.2f}%\n(n={int(row['Total'])})", ha='center', fontsize=9)

plt.tight_layout()
plt.show()
train_df.drop(columns=['night_high_vel'], inplace=True)

In [ ]:
print("\nRisk Hotspot: Foreign + Location Mismatch")

risk_pivot = train_df.pivot_table(values='is_fraud', index='foreign_transaction', columns='location_mismatch', aggfunc='mean')

plt.figure(figsize=(8, 6))
sns.heatmap(risk_pivot, annot=True, fmt='.2%', cmap='Reds', cbar_kws={'label': 'Fraud Rate'}, linewidths=0.5)
plt.title('Fraud Rate: Foreign Transaction x Location Mismatch', fontsize=14, fontweight='bold')
plt.xlabel('Location Mismatch')
plt.ylabel('Foreign Transaction')
plt.xticks([0, 1], ['Match', 'Mismatch'])
plt.yticks([0, 1], ['Domestic', 'Foreign'], rotation=0)
plt.tight_layout()
plt.show()

max_risk = risk_pivot.max().max()
min_risk = risk_pivot.min().min()
print(f"\nHighest fraud risk: {max_risk*100:.2f}% (Foreign + Mismatch)")
print(f"Lowest fraud risk: {min_risk*100:.2f}% (Domestic + Match)")
print(f"Risk multiplier: {max_risk/min_risk:.2f}x higher when both flags are active")

### Correlation Analysis of Full Engineered Feature Space
---
We evaluate linear relationships between all engineered/transformed features and the target to identify multicollinearity risks and predictive strength.

In [ ]:
all_num_cols = ['transaction_hour', 'device_trust_score', 'velocity_last_24h', 'cardholder_age', 'log_amount', 'velocity_per_hour', 'relevant_amount']
flag_cols = ['is_night_transaction', 'high_risk_abroad', 'is_high_velocity_low_trust', 'foreign_transaction', 'location_mismatch']

corr_cols = all_num_cols + flag_cols
corr_matrix = train_df[corr_cols + ['is_fraud']].corr()
target_corr = corr_matrix['is_fraud'].drop('is_fraud').sort_values(key=abs, ascending=False)

print("Feature Correlation with Target (Sorted by Absolute Value):")
print("-" * 60)
for feat, corr_val in target_corr.items():
    strength = "HIGH" if abs(corr_val) > 0.15 else "MEDIUM" if abs(corr_val) > 0.05 else "LOW"
    print(f"{strength} {feat:30s}: {corr_val:+.4f}")

plt.figure(figsize=(10, 12))
top_corr = corr_matrix[['is_fraud']].drop('is_fraud').sort_values('is_fraud', key=abs, ascending=False)
sns.heatmap(top_corr, annot=True, fmt='.3f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlation with is_fraud (Engineered Features)', fontsize=14, fontweight='bold')
plt.xlabel('Correlation Coefficient')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
selected_features = ['log_amount', 'velocity_per_hour', 'device_trust_score', 'high_risk_abroad', 'is_night_transaction', 'is_fraud']

plt.figure(figsize=(10, 8))
subset_corr = train_df[selected_features].corr()
mask = np.triu(np.ones_like(subset_corr, dtype=bool))

sns.heatmap(subset_corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0, linewidths=0.5, square=True)
plt.title('Feature Correlation Matrix (Selected Engineered Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey Correlation Insights:")
print(f"- log_amount vs is_fraud: {subset_corr.loc['log_amount', 'is_fraud']:+.3f}")
print(f"- high_risk_abroad vs is_fraud: {subset_corr.loc['high_risk_abroad', 'is_fraud']:+.3f}")
print(f"- device_trust_score vs is_fraud: {subset_corr.loc['device_trust_score', 'is_fraud']:+.3f}")
print(f"- velocity_per_hour vs is_fraud: {subset_corr.loc['velocity_per_hour', 'is_fraud']:+.3f}")

## EDA Summary & Modeling Hypotheses
---
### Key Findings Post-Engineering

1. **`log_amount`**: Successfully compresses right-skew while preserving fraud separation at higher values. Fraudulent transactions show slightly elevated log-amount distributions.

2. **`high_risk_abroad`**: Shows the strongest categorical lift (~3-4x baseline fraud rate), confirming that foreign transactions with location mismatches are highly predictive risk signals.

3. **`is_night_transaction` + `velocity_per_hour`**: The interaction reveals temporal burst patterns exclusive to fraud—legitimate users rarely exhibit high velocity during night hours.

4. **`relevant_amount`**: Amplifies outliers within specific merchant contexts, helping identify transactions that are unusually large for their category.

5. **One-hot encoded categories**: `Travel` and `Electronics` carry higher baseline fraud risk (~2.1-2.3%) than `Food` or `Clothing` (~1.2-1.4%).

6. **Correlation structure**: Engineered features show low multicollinearity (|r| < 0.3), supporting their joint use in models.

### Hypotheses Guiding Model Selection

1. **Tree-based models** (Random Forest, XGBoost, LightGBM) will outperform linear baselines due to non-linear feature interactions and threshold-based decision rules matching engineered boolean flags.

2. **Recall prioritization**: Given the business cost of missed fraud, models should optimize for recall/F1 over pure accuracy. Precision-recall curves will be more informative than ROC.

3. **Feature importance expectations**: Top predictors should be `high_risk_abroad`, `log_amount`, and `device_trust_score`.

4. **SMOTE effectiveness**: Applying SMOTE to the engineered feature space should reduce decision boundary sparsity for the minority class without distorting original distribution geometry.

5. **Calibration consideration**: Probability outputs may require calibration given the extreme class imbalance, especially for threshold-based business decisions.

## EDA Summary & Modeling Hypotheses
---
### Key Findings Post-Engineering

1. **`log_amount`**: Successfully compresses right-skew while preserving fraud separation at higher values. Fraudulent transactions show slightly elevated log-amount distributions.

2. **`high_risk_abroad`**: Shows the strongest categorical lift (~3-4x baseline fraud rate), confirming that foreign transactions with location mismatches are highly predictive risk signals.

3. **`is_night_transaction` + `velocity_per_hour`**: The interaction reveals temporal burst patterns exclusive to fraud—legitimate users rarely exhibit high velocity during night hours.

4. **`relevant_amount`**: Amplifies outliers within specific merchant contexts, helping identify transactions that are unusually large for their category.

5. **One-hot encoded categories**: `Travel` and `Electronics` carry higher baseline fraud risk (~2.1-2.3%) than `Food` or `Clothing` (~1.2-1.4%).

6. **Correlation structure**: Engineered features show low multicollinearity (|r| < 0.3), supporting their joint use in models.

### Hypotheses Guiding Model Selection

1. **Tree-based models** (Random Forest, XGBoost, LightGBM) will outperform linear baselines due to non-linear feature interactions and threshold-based decision rules matching engineered boolean flags.

2. **Recall prioritization**: Given the business cost of missed fraud, models should optimize for recall/F1 over pure accuracy. Precision-recall curves will be more informative than ROC.

3. **Feature importance expectations**: Top predictors should be `high_risk_abroad`, `log_amount`, and `device_trust_score`.

4. **SMOTE effectiveness**: Applying SMOTE to the engineered feature space should reduce decision boundary sparsity for the minority class without distorting original distribution geometry.

5. **Calibration consideration**: Probability outputs may require calibration given the extreme class imbalance, especially for threshold-based business decisions.